# Analisis Backtest MTF Trend Following (1D) & Price Action Pullback (1H) - Full Library

Dokumen ini memuat implementasi strategi kuantitatif Multi-Timeframe (MTF) murni berbasis **Trend Following (1D)** dan **Price Action Pullback (1H)** tanpa osilator CCI, yang direfaktor secara penuh menggunakan library standar industri: **pandas-ta** untuk kalkulasi teknikal dan **Backtesting.py** untuk mesin eksekusi simulasi, perhitungan metrik performa, serta visualisasi grafik interaktif.

Arsitektur Sistem & Penyelarasan Mekanis:
1. **Parameter Universal**: Seluruh instrumen diuji menggunakan konfigurasi parameter seragam (`GLOBAL_PARAMS`) tanpa override lokal per simbol.
2. **Sanitasi Jam Bursa Reguler (RTH)**: Data 1H disaring hanya pada jam bursa reguler (09:00 - 16:00 waktu bursa) dan bebas dari bar akhir pekan (Sabtu/Minggu).
3. **Penyelarasan Bar Awal (Eliminasi Warmup Desync)**: Periode pemanasan indikator diselaraskan sehingga bar transaksi pertama yang dievaluasi oleh Backtesting.py persis sama dengan mesin native.
4. **Pencegahan Lookahead Bias**: Seluruh indikator makro 1D (`EMA_fast`, `EMA_slow`) digeser satu bar (`.shift(1)`) sebelum digabungkan (*forward-fill*) ke data 1H.
5. **Kalkulasi Indikator Baku**:
   - `EMA_fast` (1D): periode 20 hari
   - `EMA_slow` (1D): periode 50 hari
   - `macro_bullish = EMA_fast > EMA_slow`, `macro_bearish = EMA_fast < EMA_slow`
   - `EMA_micro` (1H): periode 32 bar (level harga wajar / target pullback)
   - `ATR` (1H): periode 7 bar dengan penghalusan eksponensial untuk trailing stop adaptif.
6. **Eksekusi Trailing Stop Intrabar Selaras (Backtesting.py Native SL)**:
   - Order masuk dengan level Stop Loss awal (`self.buy(sl=initial_sl, size=0.9999)` / `self.sell(sl=initial_sl, size=0.9999)`).
   - Trailing stop diperbarui per bar secara adaptif melalui `trade.sl = new_sl`, mengeksekusi exit intrabar tepat pada harga SL ketika Low/High menembus batas stop, mengeliminasi bias eksekusi Close bar.
   - Posisi ditutup secara diskresioner (`self.position.close()`) hanya ketika tren makro 1D berbalik arah (*Macro Trend Broken*).
7. **Jeda Cooldown Transaksi**:
   - Pelacakan jeda bar (`bars_since_exit >= 6`) diselaraskan dengan mesin native setelah posisi ditutup.
8. **Visualisasi Interaktif Bokeh Inline**:
   - Visualisasi interaktif dirender langsung di dalam log notebook (`open_browser=False`) tanpa membuat file HTML terpisah.

In [113]:
# =============================================================================
# KONFIGURASI GLOBAL & PARAMETER UNIVERSAL
# =============================================================================
CONFIG = {
    # --- Sumber data ---------------------------------------------------------
    'data_dir': 'data_index',
    'instruments': {
        'SPY':  {'nama': 'S&P 500 (SPY)',    'file_1d': 'spy_US500_D1.parquet',
                 'file_1h': 'spy_US500_H1.parquet', 'warna': 'blue'},
        'QQQ':  {'nama': 'NASDAQ 100 (QQQ)', 'file_1d': 'Nasdaq100_1d_data.csv',
                 'file_1h': 'Nasdaq100_1h_data.csv', 'warna': 'green'},
        'DJI':  {'nama': 'DOW JONES (DJI)',  'file_1d': 'Dow_Jones_1d_data.csv',
                 'file_1h': 'Dow_Jones_1h_data.csv', 'warna': 'purple'},
        'AAPL': {'nama': 'APPLE INC. (AAPL)', 'file_1d': 'validasi-aapl/aapl_1d.csv',
                 'file_1h': 'validasi-aapl/aapl_1h.csv', 'warna': 'darkorange'},
    },

    # --- Pra-pemrosesan data -------------------------------------------------
    'sinkronisasi_rentang': True,   # potong ke irisan waktu bersama semua instrumen
    'hapus_akhir_pekan': True,      # buang baris Sabtu/Minggu (artefak data 24/7)
    'filter_rth': True,             # hanya simpan jam bursa reguler (09:00 - 16:00)

    # --- Sistem modal & biaya ------------------------------------------------
    'initial_capital': 1000.0,
    'fee': 0.0003,                  # komisi 0.02% + slippage 0.01% per sisi transaksi

    # --- Pelaporan -----------------------------------------------------------
    'trading_days_per_year': 252,
    'fase_pasar': {
        'urutan': ['Bullish', 'Bearish', 'Sideways'],
        'default': 'Sideways',
        'label_keseluruhan': 'Keseluruhan Berjalan',
    },
}

GLOBAL_PARAMS = {
    'initial_capital': 1000.0,
    'fee': 0.0003,
    'ema_macro_fast': 20,          # EMA 1D arah tren utama
    'ema_macro_slow': 50,          # EMA 1D jangkar tren utama
    'ema_micro': 32,               # EMA 1H sebagai level harga wajar (pullback target)
    'atr_period': 7,              # ATR 1H untuk trailing stop adaptif
    'atr_mult_long': 5.0,         # Trailing stop lebar untuk Semi Buy & Hold Long (10x ATR)
    'atr_mult_short': 3.0,         # Trailing stop ketat untuk pengamanan cepat Short (2.5x ATR)
    'cooldown_bars': 14,            # Jeda normal 6 bar (~1 hari bursa)
    'enable_short': True,          # Aktifkan perdagangan dua arah (Long & Short)
    'macro_spread_bull': 0.002,    # Spread Long cukup 0.2% (responsif menangkap tren naik)
    'macro_spread_bear': 0.01,    # Spread Short wajib -0.5% (selektif hanya saat tren turun kuat)
    'min_profit_pct_long': 0.02   # Batas profit +0.5% sebelum trailing stop Long diizinkan menutup posisi
}

print("Konfigurasi global & parameter asimetris dimuat:", list(CONFIG['instruments'].keys()))


Konfigurasi global & parameter asimetris dimuat: ['SPY', 'QQQ', 'DJI', 'AAPL']


In [114]:
%pip install pandas numpy matplotlib seaborn pandas_ta backtesting bokeh


Note: you may need to restart the kernel to use updated packages.


In [115]:
# =============================================================================
# BLOK 1: PERSIAPAN & LOAD LIBRARY
# =============================================================================
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
# Monkey-patch untuk kompatibilitas Pandas 2.x/3.x
pd.DataFrame.iteritems = pd.DataFrame.items
import numpy as np
import pandas_ta as ta
from bokeh.io import output_notebook
from backtesting import Backtest, Strategy, set_bokeh_output
from backtesting.lib import FractionalBacktest

# Aktifkan output visualisasi interaktif Bokeh langsung ke notebook (inline)
output_notebook()
set_bokeh_output(notebook=True)

print("Seluruh library baku (pandas, numpy, pandas-ta, Backtesting.py, bokeh) berhasil dimuat.")

Loading BokehJS ...

Seluruh library baku (pandas, numpy, pandas-ta, Backtesting.py, bokeh) berhasil dimuat.


In [116]:
# =============================================================================
# BLOK 2: INGESTI, PEMBERSIHAN, & SINKRONISASI RENTANG WAKTU
# =============================================================================
def load_and_clean_data(file_path):
    if file_path.endswith('.parquet'):
        df = pd.read_parquet(file_path)
    else:
        with open(file_path) as f:
            sep = '\t' if '\t' in f.readline() else ','
        df = pd.read_csv(file_path, sep=sep)

    df.columns = [c.lower() for c in df.columns]

    if 'datetime' in df.columns:
        df['datetime'] = pd.to_datetime(df['datetime'])
    elif 'date' in df.columns:
        df['datetime'] = pd.to_datetime(df['date'])
        df.drop(columns=['date'], inplace=True, errors='ignore')
    elif 'time' in df.columns:
        df['datetime'] = pd.to_datetime(df['time'])

    df.sort_values('datetime', inplace=True)
    df.set_index('datetime', inplace=True)

    if CONFIG['hapus_akhir_pekan']:
        df = df[df.index.dayofweek < 5]

    df['date'] = df.index.date
    return df

def muat_semua_data(config):
    data = {}
    for kode, meta in config['instruments'].items():
        df_1d = load_and_clean_data(os.path.join(config['data_dir'], meta['file_1d']))
        df_1h = load_and_clean_data(os.path.join(config['data_dir'], meta['file_1h']))
        
        # Filter Jam Bursa Reguler / RTH (09:00 - 16:00 waktu bursa)
        if config.get('filter_rth', True):
            df_1h = df_1h[(df_1h.index.hour >= 9) & (df_1h.index.hour <= 16)]
            
        data[kode] = {'1d': df_1d, '1h': df_1h}
    return data

def hitung_rentang_bersama(data):
    awal = [df.index[0].normalize() for d in data.values() for df in d.values()]
    akhir = [df.index[-1].normalize() for d in data.values() for df in d.values()]
    start, end = max(awal), min(akhir)
    if start > end:
        raise ValueError(f"Tidak ada irisan waktu antar instrumen: {start} > {end}")
    return start, end

def potong_rentang(df, start, end):
    return df.loc[start: end + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)].copy()

DATA_MENTAH = muat_semua_data(CONFIG)

if CONFIG['sinkronisasi_rentang']:
    START_BERSAMA, END_BERSAMA = hitung_rentang_bersama(DATA_MENTAH)
    DATA = {k: {tf: potong_rentang(df, START_BERSAMA, END_BERSAMA) for tf, df in d.items()}
            for k, d in DATA_MENTAH.items()}
else:
    START_BERSAMA, END_BERSAMA = None, None
    DATA = DATA_MENTAH

print("Data historis berhasil dimuat dan disanitasi (RTH 09:00-16:00).")
if CONFIG['sinkronisasi_rentang']:
    print(f"Rentang waktu bersama: {START_BERSAMA.date()} s.d. {END_BERSAMA.date()}")
print(f"{'Instrumen':<10}{'TF':<5}{'Baris (mentah)':>16}{'Baris (sinkron)':>17}  Rentang sinkron")
for kode in DATA:
    for tf in ('1d', '1h'):
        raw, syn = DATA_MENTAH[kode][tf], DATA[kode][tf]
        print(f"{kode:<10}{tf.upper():<5}{len(raw):>16,}{len(syn):>17,}  "
              f"{syn.index[0]} -> {syn.index[-1]}")

Data historis berhasil dimuat dan disanitasi (RTH 09:00-16:00).
Rentang waktu bersama: 2018-02-19 s.d. 2024-05-10
Instrumen TF     Baris (mentah)  Baris (sinkron)  Rentang sinkron
SPY       1D              3,627            1,608  2018-02-19 00:00:00 -> 2024-05-10 00:00:00
SPY       1H             17,525           12,829  2018-02-19 16:00:00 -> 2024-05-10 16:00:00
QQQ       1D              2,283            1,600  2018-02-19 00:00:00 -> 2024-05-10 00:00:00
QQQ       1H             18,209           12,769  2018-02-19 09:00:00 -> 2024-05-10 16:00:00
DJI       1D              2,250            1,609  2018-02-19 00:00:00 -> 2024-05-10 00:00:00
DJI       1H             17,978           12,850  2018-02-19 09:00:00 -> 2024-05-10 16:00:00
AAPL      1D              4,384            1,485  2018-02-19 00:00:00 -> 2024-05-10 00:00:00
AAPL      1H             35,072           11,880  2018-02-19 09:00:00 -> 2024-05-10 16:00:00


In [117]:
# =============================================================================
# BLOK 3: FEATURE ENGINEERING MULTI-TIMEFRAME & PENYELARASAN MTF
# =============================================================================
def calc_ema_series(series, span):
    return series.ewm(span=span, adjust=False).mean()

def calc_atr_series(high, low, close, period=14):
    high_s = pd.Series(high)
    low_s = pd.Series(low)
    close_s = pd.Series(close)
    close_prev = close_s.shift(1)
    tr = pd.concat([high_s - low_s,
                    (high_s - close_prev).abs(),
                    (low_s - close_prev).abs()], axis=1).max(axis=1)
    return calc_ema_series(tr, period)

def persiapkan_data_multitimeframe(df_1d_raw, df_1h_raw, params=None):
    p = GLOBAL_PARAMS if params is None else params
    
    # 1. Feature Engineering Timeframe 1D (Tren Makro Asimetris dengan Shift 1 Bar)
    d1 = df_1d_raw.copy()
    d1['close_prev'] = d1['close'].shift(1)
    d1['ema_fast'] = calc_ema_series(d1['close'], p['ema_macro_fast']).shift(1)
    d1['ema_fast_prev'] = d1['ema_fast'].shift(1)
    d1['ema_slow'] = calc_ema_series(d1['close'], p['ema_macro_slow']).shift(1)
    d1['ema_spread'] = (d1['ema_fast'] - d1['ema_slow']) / d1['ema_slow']
    
    # Syarat Izin Entry Makro (Asimetris):
    # Long (Moderat): spread > +0.2% dan kemiringan EMA fast harian datar/naik
    d1['macro_entry_bull'] = (d1['ema_spread'] > p['macro_spread_bull']) & (d1['ema_fast'] >= d1['ema_fast_prev'])
    # Short (Ketat): spread < -0.5% dan close kemarin di bawah EMA fast dan EMA fast miring turun
    d1['macro_entry_bear'] = (d1['ema_spread'] < -p['macro_spread_bear']) & (d1['close_prev'] < d1['ema_fast']) & (d1['ema_fast'] < d1['ema_fast_prev'])
    
    # Syarat Tren Makro Masih Hidup (Hard-Exit)
    d1['macro_bull_alive'] = d1['ema_fast'] >= d1['ema_slow']
    d1['macro_bear_alive'] = d1['ema_fast'] <= d1['ema_slow']
    
    # Klasifikasi Label fase_pasar
    cfg = CONFIG['fase_pasar']
    bullish, bearish, sideways = cfg['urutan']
    fase = pd.Series(sideways, index=d1.index, dtype=object)
    fase[d1['macro_entry_bull']] = bullish
    fase[d1['macro_entry_bear']] = bearish
    d1['fase_pasar'] = fase
    
    # 2. Penyelarasan MTF 1D ke 1H berbasis tanggal
    d1_sub = d1[['date', 'ema_fast', 'ema_slow', 'ema_spread', 'macro_entry_bull', 'macro_entry_bear', 'macro_bull_alive', 'macro_bear_alive', 'fase_pasar']]
    df = pd.merge(df_1h_raw.copy().reset_index(), d1_sub, on='date', how='left')
    df.ffill(inplace=True)
    df.set_index('datetime', inplace=True)
    
    # 3. Standardisasi format kolom sesuai spesifikasi Backtesting.py
    kolom_standar = {
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close',
        'volume': 'Volume'
    }
    df.rename(columns=kolom_standar, inplace=True)
    
    # 4. Feature Engineering Timeframe 1H (EMA Micro & ATR)
    df['EMA_micro'] = calc_ema_series(df['Close'], p['ema_micro'])
    df['ATR'] = calc_atr_series(df['High'], df['Low'], df['Close'], p['atr_period'])
    
    # Drop baris NaN awal
    df.dropna(inplace=True)
    
    return df

DATA_PREPARED = {}
for sym in CONFIG['instruments']:
    DATA_PREPARED[sym] = persiapkan_data_multitimeframe(DATA[sym]['1d'], DATA[sym]['1h'])
    print(f"Data {sym} siap: {len(DATA_PREPARED[sym]):,} bar 1H | Rentang: {DATA_PREPARED[sym].index[0]} s.d. {DATA_PREPARED[sym].index[-1]}")


Data SPY siap: 12,828 bar 1H | Rentang: 2018-02-20 09:00:00 s.d. 2024-05-10 16:00:00
Data QQQ siap: 12,761 bar 1H | Rentang: 2018-02-20 09:00:00 s.d. 2024-05-10 16:00:00
Data DJI siap: 12,842 bar 1H | Rentang: 2018-02-20 09:00:00 s.d. 2024-05-10 16:00:00
Data AAPL siap: 11,872 bar 1H | Rentang: 2018-02-20 09:00:00 s.d. 2024-05-10 16:00:00


In [118]:
# =============================================================================
# BLOK 4: ARSITEKTUR STRATEGI SELARAS DENGAN BACKTESTING.PY
# =============================================================================
def hitung_ema_micro(close, span=32):
    return calc_ema_series(pd.Series(close), span).values

def hitung_atr_micro(high, low, close, period=14):
    return calc_atr_series(high, low, close, period).values

class MTFPullbackStrategy(Strategy):
    atr_mult_long = GLOBAL_PARAMS['atr_mult_long']
    atr_mult_short = GLOBAL_PARAMS['atr_mult_short']
    cooldown_bars = GLOBAL_PARAMS['cooldown_bars']
    enable_short = GLOBAL_PARAMS['enable_short']
    min_profit_pct_long = GLOBAL_PARAMS['min_profit_pct_long']

    def init(self):
        # Mendaftarkan indikator menggunakan self.I() agar terpantau engine dan masuk plot otomatis
        self.ema_micro = self.I(hitung_ema_micro, self.data.Close, GLOBAL_PARAMS['ema_micro'], name='EMA_micro', overlay=True)
        self.atr = self.I(hitung_atr_micro, self.data.High, self.data.Low, self.data.Close, GLOBAL_PARAMS['atr_period'], name='ATR', overlay=False)
        self.macro_entry_bull = self.I(lambda: self.data.df['macro_entry_bull'].values, name='macro_entry_bull', overlay=False)
        self.macro_entry_bear = self.I(lambda: self.data.df['macro_entry_bear'].values, name='macro_entry_bear', overlay=False)
        self.macro_bull_alive = self.I(lambda: self.data.df['macro_bull_alive'].values, name='macro_bull_alive', overlay=False)
        self.macro_bear_alive = self.I(lambda: self.data.df['macro_bear_alive'].values, name='macro_bear_alive', overlay=False)
        
        # Inisialisasi status pelacak transaksi
        self.bars_since_exit = 999
        self.trailing_sl = 0.0
        self.entry_price = 0.0

    def next(self):
        if len(self.data) < 5:
            return

        # 1. Update counter cooldown jika tidak ada posisi aktif
        if not self.position:
            self.bars_since_exit += 1

            # BLOK ENTRY: Dievaluasi saat not self.position dan bars_since_exit >= cooldown_bars
            if self.bars_since_exit >= self.cooldown_bars:
                # a. Entry LONG (Semi Buy & Hold - Pemicu Pullback Sederhana):
                long_trigger = (
                    self.macro_entry_bull[-1] and 
                    (self.data.Low[-2] <= self.ema_micro[-2]) and 
                    (self.data.Close[-1] > self.ema_micro[-1]) and 
                    (self.ema_micro[-1] >= self.ema_micro[-2])
                )

                # b. Entry SHORT (Strict Tactical - Pemicu Rejection Detail):
                short_trigger = (
                    self.enable_short and 
                    self.macro_entry_bear[-1] and 
                    (self.data.High[-2] >= self.ema_micro[-2]) and 
                    (self.data.Close[-1] < self.ema_micro[-1]) and 
                    (self.data.Close[-1] < self.data.Low[-2]) and 
                    (self.ema_micro[-1] < self.ema_micro[-2]) and 
                    (self.data.index[-1].hour != 9)
                )

                if long_trigger:
                    self.buy(size=0.9999)
                    self.entry_price = self.data.Close[-1]
                    self.trailing_sl = self.data.Close[-1] - (self.atr_mult_long * self.atr[-1])
                    return
                elif short_trigger:
                    self.sell(size=0.9999)
                    self.entry_price = self.data.Close[-1]
                    self.trailing_sl = self.data.Close[-1] + (self.atr_mult_short * self.atr[-1])
                    return

        # 2. BLOK MANAJEMEN POSISI & EXIT
        else:
            if self.position.is_long:
                # Update Ratchet Stop Lebar (Semi Buy & Hold)
                new_sl = self.data.Close[-1] - (self.atr_mult_long * self.atr[-1])
                self.trailing_sl = max(self.trailing_sl, new_sl)
                
                # Cek apakah posisi sudah berada di zona profit aman (+0.5% di atas harga beli)
                is_in_profit = self.data.Close[-1] > (self.entry_price * (1.0 + self.min_profit_pct_long))
                
                # Exit 1 (Trailing Stop HANYA jika posisi sudah cuan dan di luar jam 09:00):
                if self.data.index[-1].hour != 9 and is_in_profit and self.data.Close[-1] <= self.trailing_sl:
                    self.position.close()
                    self.bars_since_exit = 0
                    return
                
                # Exit 2 (Cut Loss / Hard-Exit HANYA jika tren makro 1D benar-benar patah EMA 20 < EMA 50):
                if not self.macro_bull_alive[-1]:
                    self.position.close()
                    self.bars_since_exit = 0
                    return

            elif self.position.is_short:
                # Update Ratchet Stop Ketat (Tactical Risk Control)
                new_sl = self.data.Close[-1] + (self.atr_mult_short * self.atr[-1])
                self.trailing_sl = min(self.trailing_sl, new_sl)
                
                # Exit 1 (Trailing Stop langsung aktif mengawal posisi di luar jam 09:00, baik untung maupun rugi):
                if self.data.index[-1].hour != 9 and self.data.Close[-1] >= self.trailing_sl:
                    self.position.close()
                    self.bars_since_exit = 0
                    return
                
                # Exit 2 (Hard-Exit jika tren makro Bearish patah EMA 20 > EMA 50):
                if not self.macro_bear_alive[-1]:
                    self.position.close()
                    self.bars_since_exit = 0
                    return

print("Class MTFPullbackStrategy dengan arsitektur Asimetris Long-Short berhasil didefinisikan.")


Class MTFPullbackStrategy dengan arsitektur Asimetris Long-Short berhasil didefinisikan.


In [119]:
# =============================================================================
# BLOK 5: MESIN EVALUASI FASE PASAR & EKSTRAKSI METRIK BAWAAN
# =============================================================================
METRIK_KUNCI = [
    'Return [%]',
    'Buy & Hold Return [%]',
    'Return (Ann.) [%]',
    'Max. Drawdown [%]',
    'Sharpe Ratio',
    'Sortino Ratio',
    'Win Rate [%]',
    '# Trades'
]

def buat_instance_backtest(df, strategy=MTFPullbackStrategy):
    # Menggunakan FractionalBacktest untuk mendukung continuous fractional compounding
    return FractionalBacktest(
        df,
        strategy,
        cash=CONFIG['initial_capital'],
        commission=CONFIG['fee'],
        trade_on_close=True,
        finalize_trades=True,
        fractional_unit=1e-4
    )

def format_nilai_metrik(val):
    if pd.isna(val) or val == '--':
        return '-'
    if isinstance(val, (int, np.integer)):
        return str(val)
    if isinstance(val, (float, np.floating)):
        return f"{val:.2f}"
    return str(val)

def hitung_metrik_fase(trades_sub, bnh_return, total_duration_years=None):
    if len(trades_sub) == 0:
        return {
            'Return [%]': '0.00',
            'Buy & Hold Return [%]': format_nilai_metrik(bnh_return),
            'Return (Ann.) [%]': '0.00',
            'Max. Drawdown [%]': '-0.00',
            'Sharpe Ratio': '-',
            'Sortino Ratio': '-',
            'Win Rate [%]': '-',
            '# Trades': '0'
        }
    
    n_trades = len(trades_sub)
    win_rate = (trades_sub['ReturnPct'] > 0).mean() * 100
    comp_ret = ((1 + trades_sub['ReturnPct']).prod() - 1)
    ret_pct = comp_ret * 100
    
    if total_duration_years and total_duration_years > 0:
        ann_ret = (((1 + comp_ret)**(1 / total_duration_years) - 1) * 100) if comp_ret > -1.0 else -100.0
    else:
        ann_ret = np.nan
        
    eq_curve = 1000.0 * np.cumprod(np.insert(1 + trades_sub['ReturnPct'].values, 0, 1.0))
    peaks = np.maximum.accumulate(eq_curve)
    dd = 1.0 - eq_curve / peaks
    max_dd = -dd.max() * 100
    
    r_vals = trades_sub['ReturnPct'].values
    r_mean = np.mean(r_vals)
    r_std = np.std(r_vals, ddof=1) if len(r_vals) > 1 else np.nan
    
    trades_per_year = n_trades / (total_duration_years if total_duration_years else 1.0)
    sharpe = (r_mean / r_std * np.sqrt(trades_per_year)) if (r_std and r_std > 0) else np.nan
    
    downside_r = r_vals[r_vals < 0]
    downside_std = np.sqrt(np.mean(downside_r**2)) if len(downside_r) > 0 else 0
    sortino = (r_mean / downside_std * np.sqrt(trades_per_year)) if downside_std > 0 else np.nan
    
    return {
        'Return [%]': format_nilai_metrik(ret_pct),
        'Buy & Hold Return [%]': format_nilai_metrik(bnh_return),
        'Return (Ann.) [%]': format_nilai_metrik(ann_ret),
        'Max. Drawdown [%]': format_nilai_metrik(max_dd),
        'Sharpe Ratio': format_nilai_metrik(sharpe),
        'Sortino Ratio': format_nilai_metrik(sortino),
        'Win Rate [%]': format_nilai_metrik(win_rate),
        '# Trades': str(n_trades)
    }

def evaluasi_lengkap_instrumen(df_prepared, symbol_name):
    # 1. Backtest Penuh Keseluruhan (Continuous Run)
    bt_all = buat_instance_backtest(df_prepared)
    stats_all = bt_all.run()
    
    tabel_laporan = []
    baris_all = {'Fase Pasar': CONFIG['fase_pasar']['label_keseluruhan']}
    for m in METRIK_KUNCI:
        baris_all[m] = format_nilai_metrik(stats_all.get(m, np.nan))
    tabel_laporan.append(baris_all)
    
    # 2. Pemetaan Transaksi Berdasarkan Rezim Entry (Mencegah Regime Transition Leakage)
    trades = stats_all['_trades'].copy()
    total_duration_years = (df_prepared.index[-1] - df_prepared.index[0]).total_seconds() / (365.25 * 86400)
    bnh_overall = stats_all.get('Buy & Hold Return [%]', np.nan)
    
    if len(trades) > 0:
        trades['fase_entry'] = [
            df_prepared.at[t, 'fase_pasar'] if t in df_prepared.index else df_prepared['fase_pasar'].iloc[b]
            for t, b in zip(trades['EntryTime'], trades['EntryBar'])
        ]
    else:
        trades['fase_entry'] = pd.Series(dtype=object)
        
    for fase in CONFIG['fase_pasar']['urutan']:
        trades_fase = trades[trades['fase_entry'] == fase] if len(trades) > 0 else pd.DataFrame()
        
        df_fase = df_prepared[df_prepared['fase_pasar'] == fase]
        if len(df_fase) > 1:
            bnh_fase = (df_fase['Close'].iloc[-1] - df_fase['Close'].iloc[0]) / df_fase['Close'].iloc[0] * 100
        else:
            bnh_fase = bnh_overall
            
        metrik_fase = hitung_metrik_fase(trades_fase, bnh_fase, total_duration_years)
        baris_fase = {'Fase Pasar': fase}
        baris_fase.update(metrik_fase)
        tabel_laporan.append(baris_fase)
        
    df_laporan = pd.DataFrame(tabel_laporan)
    
    # 3. Cetak Ringkasan Performa
    print("=" * 100)
    print(f"HASIL EVALUASI BACKTESTING.PY - [{symbol_name.upper()}]")
    print(f"Periode Simulasi : {df_prepared.index[0]:%Y-%m-%d %H:%M} s.d. {df_prepared.index[-1]:%Y-%m-%d %H:%M} ({len(df_prepared):,} bar 1H)")
    print(f"Engine Type      : {type(bt_all).__name__}")
    print("=" * 100)
    print(df_laporan.to_string(index=False))
    print("\n")
    
    return bt_all, stats_all, df_laporan


In [120]:
# =============================================================================
# BLOK 6: EKSEKUSI SELURUH INSTRUMEN, VISUALISASI INLINE & TABEL KOMPARASI
# =============================================================================
HASIL_BACKTEST = {}

for sym in CONFIG['instruments']:
    meta = CONFIG['instruments'][sym]
    df_prep = DATA_PREPARED[sym]
    
    # Eksekusi Evaluasi Backtest
    bt_obj, stats_obj, df_lap = evaluasi_lengkap_instrumen(df_prep, meta['nama'])
    HASIL_BACKTEST[sym] = {
        'bt': bt_obj,
        'stats': stats_obj,
        'laporan': df_lap
    }
    
    # Pembuatan Visualisasi Interaktif Langsung di Output Notebook (Inline Bokeh)
    print(f"Menampilkan visualisasi interaktif Bokeh inline untuk [{sym}]...")
    try:
        bt_obj.plot(open_browser=False)
    except Exception as e:
        print(f"Peringatan pembuatan visualisasi {sym}: {e}")
    print("-" * 100)

# =============================================================================
# TABEL KOMPARATIF KINERJA STRATEGI ANTAR INSTRUMEN
# =============================================================================
ringkasan_komparatif = []

for sym in CONFIG['instruments']:
    meta = CONFIG['instruments'][sym]
    stats = HASIL_BACKTEST[sym]['stats']
    bnh = stats.get('Buy & Hold Return [%]', stats.get('Buy & Hold [%]', np.nan))
    
    baris = {
        'Instrumen': meta['nama'],
        'Return [%]': format_nilai_metrik(stats.get('Return [%]', np.nan)),
        'Buy & Hold [%]': format_nilai_metrik(bnh),
        'Return (Ann.) [%]': format_nilai_metrik(stats.get('Return (Ann.) [%]', np.nan)),
        'Max. Drawdown [%]': format_nilai_metrik(stats.get('Max. Drawdown [%]', np.nan)),
        'Sharpe Ratio': format_nilai_metrik(stats.get('Sharpe Ratio', np.nan)),
        'Sortino Ratio': format_nilai_metrik(stats.get('Sortino Ratio', np.nan)),
        'Win Rate [%]': format_nilai_metrik(stats.get('Win Rate [%]', np.nan)),
        '# Trades': format_nilai_metrik(stats.get('# Trades', 0))
    }
    ringkasan_komparatif.append(baris)

df_komparatif = pd.DataFrame(ringkasan_komparatif)

print("=" * 110)
print("TABEL PERBANDINGAN KINERJA MULTI-TIMEFRAME ANTAR INSTRUMEN")
print("=" * 110)
print(df_komparatif.to_string(index=False))
print("=" * 110)


FractionalBacktest.run:   0%|          | 0/12827 [00:00<?, ?bar/s]

HASIL EVALUASI BACKTESTING.PY - [S&P 500 (SPY)]
Periode Simulasi : 2018-02-20 09:00 s.d. 2024-05-10 16:00 (12,828 bar 1H)
Engine Type      : FractionalBacktest
          Fase Pasar Return [%] Buy & Hold Return [%] Return (Ann.) [%] Max. Drawdown [%] Sharpe Ratio Sortino Ratio Win Rate [%] # Trades
Keseluruhan Berjalan      68.48                 92.18              8.53            -27.40         0.61          0.96        60.32       63
             Bullish      46.00                 90.13              6.27            -19.92         0.63          0.44        74.42       43
             Bearish      15.43                 65.69              2.33            -12.57         0.31          1.08        30.00       20
            Sideways       0.00                 88.21              0.00             -0.00            -             -            -        0


Menampilkan visualisasi interaktif Bokeh inline untuk [SPY]...


----------------------------------------------------------------------------------------------------


FractionalBacktest.run:   0%|          | 0/12760 [00:00<?, ?bar/s]

HASIL EVALUASI BACKTESTING.PY - [NASDAQ 100 (QQQ)]
Periode Simulasi : 2018-02-20 09:00 s.d. 2024-05-10 16:00 (12,761 bar 1H)
Engine Type      : FractionalBacktest
          Fase Pasar Return [%] Buy & Hold Return [%] Return (Ann.) [%] Max. Drawdown [%] Sharpe Ratio Sortino Ratio Win Rate [%] # Trades
Keseluruhan Berjalan     109.58                169.18             12.38            -18.70         0.72          1.17        67.16       67
             Bullish      98.46                159.81             11.65            -17.58         1.03          0.78        76.09       46
             Bearish       5.66                126.89              0.89             -7.25         0.19          0.24        47.62       21
            Sideways       0.00                169.18              0.00             -0.00            -             -            -        0


Menampilkan visualisasi interaktif Bokeh inline untuk [QQQ]...


...

HASIL EVALUASI BACKTESTING.PY - [DOW JONES (DJI)]
Periode Simulasi : 2018-02-20 09:00 s.d. 2024-05-10 16:00 (12,842 bar 1H)
Engine Type      : FractionalBacktest
          Fase Pasar Return [%] Buy & Hold Return [%] Return (Ann.) [%] Max. Drawdown [%] Sharpe Ratio Sortino Ratio Win Rate [%] # Trades
Keseluruhan Berjalan       9.47                 57.53              1.43            -20.11         0.12          0.18        50.00       54
             Bullish      -6.60                 57.57             -1.09            -17.81        -0.06         -0.05        60.00       35
             Bearish      17.20                 39.11              2.58             -6.20         0.31          1.18        31.58       19
            Sideways       0.00                 57.53              0.00             -0.00            -             -            -        0


Menampilkan visualisasi interaktif Bokeh inline untuk [DJI]...


----------------------------------------------------------------------------------------------------


FractionalBacktest.run:   0%|          | 0/11871 [00:00<?, ?bar/s]

HASIL EVALUASI BACKTESTING.PY - [APPLE INC. (AAPL)]
Periode Simulasi : 2018-02-20 09:00 s.d. 2024-05-10 16:00 (11,872 bar 1H)
Engine Type      : FractionalBacktest
          Fase Pasar Return [%] Buy & Hold Return [%] Return (Ann.) [%] Max. Drawdown [%] Sharpe Ratio Sortino Ratio Win Rate [%] # Trades
Keseluruhan Berjalan     271.46                328.65             24.98            -21.79         1.29          2.59        60.00       15
             Bullish     271.62                326.30             23.50            -14.28         0.86          2.88        60.00       15
             Bearish       0.00                262.03              0.00             -0.00            -             -            -        0
            Sideways       0.00                328.65              0.00             -0.00            -             -            -        0


Menampilkan visualisasi interaktif Bokeh inline untuk [AAPL]...


----------------------------------------------------------------------------------------------------
TABEL PERBANDINGAN KINERJA MULTI-TIMEFRAME ANTAR INSTRUMEN
        Instrumen Return [%] Buy & Hold [%] Return (Ann.) [%] Max. Drawdown [%] Sharpe Ratio Sortino Ratio Win Rate [%] # Trades
    S&P 500 (SPY)      68.48          92.18              8.53            -27.40         0.61          0.96        60.32       63
 NASDAQ 100 (QQQ)     109.58         169.18             12.38            -18.70         0.72          1.17        67.16       67
  DOW JONES (DJI)       9.47          57.53              1.43            -20.11         0.12          0.18        50.00       54
APPLE INC. (AAPL)     271.46         328.65             24.98            -21.79         1.29          2.59        60.00       15
